In [1]:
import src.database.scripts.sql as sql
from src.database.scrappers.recipes_fetch import merchant_recipes

In [ ]:
def drop_recipes_table():
    query = "DROP TABLE IF EXISTS season_8.recipes"
    cursor.execute(query)

def create_recipes_table():
    query = """
    CREATE TABLE IF NOT EXISTS recipes(
    recipe_id   INT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    name        TEXT,
    amount      INT,
    rarity      TEXT,
    merchant    TEXT,
    affinity    INT
    )"""
    cursor.execute(query)

def recipes_fetch():
    recipes_list = []
    for merchant in merchant_recipes.merchant_list:
        print(f'parsing {merchant}')
        table = merchant_recipes(merchant)
        for row in table.row_range():
            table.row_target(row)
            amount, rarity, name = table.item()
            merchant, affinity = table.vendor()
            recipes_list.append((name, amount, rarity, merchant, affinity))
    return recipes_list

def insert_recipes_table():
    row_list = recipes_fetch() 
    query = """
    INSERT INTO season_8.recipes (name, amount, rarity, merchant, affinity)
    VALUES (%s, %s, %s, %s, %s)
    """
    cursor.executemany(query, row_list)

In [58]:
def drop_ingredients_table():
    query = "DROP TABLE IF EXISTS season_8.ingredients"
    cursor.execute(query)

def create_ingredients_table():
    query = """
    CREATE TABLE IF NOT EXISTS ingredients(
    recipe_id   INT NOT NULL REFERENCES season_8.recipes(recipe_id),
    name        TEXT,
    amount      INT,  
    rarity      TEXT,
    PRIMARY KEY (recipe_id, name, rarity)
    )"""
    cursor.execute(query)

def ingredients_fetch():
    ingredients_list = []
    recipe_id = 1
    for merchant in merchant_recipes.merchant_list:
        print(f'parsing {merchant}')
        table = merchant_recipes(merchant)
        for row in table.row_range():
            table.row_target(row)
            ingredients = table.ingredients()
            ingredients = [(recipe_id,) + t for t in ingredients]
            ingredients_list.extend(ingredients)
            recipe_id += 1
    return ingredients_list

def insert_ingredients_table():
    ingredients_list = ingredients_fetch()
    query = """
    INSERT TABLE season_8.ingredients (recipe_id, name, amoutn, rarity)
    VALUES (%s, %s, %s, %s)
    """
    cursor.executemany(query, ingredients_list)

In [55]:
test = ingredients_fetch()
test

parsing Alchemist
parsing Armourer
parsing Goldsmith
parsing Leathersmith
parsing Tailor
parsing Weaponsmith
parsing Woodsman


[(1, '2', 'Common', 'Iron Ore'),
 (2, '2', 'Uncommon', 'Copper Ore'),
 (3, '2', 'Rare', 'Cobalt Ore'),
 (4, '2', 'Epic', 'Rubysilver Ore'),
 (5, '45', 'Common', 'Silver Coin'),
 (6, '2', 'Epic', 'Gold Ore'),
 (7, '2', 'Epic', 'Froststone Ore'),
 (8, '2', 'Epic', 'Tidestone Ore'),
 (9, '2', 'Epic', 'Obsidian Ore'),
 (10, '1', 'Common', 'Lifeleaf'),
 (11, '1', 'Uncommon', 'Lifeleaf'),
 (12, '1', 'Rare', 'Lifeleaf'),
 (13, '5', 'Uncommon', 'Lifeleaf'),
 (14, '1', 'Common', 'Phantom Flower'),
 (15, '1', 'Uncommon', 'Phantom Flower'),
 (16, '1', 'Rare', 'Phantom Flower'),
 (17, '5', 'Uncommon', 'Phantom Flower'),
 (18, '1', 'Common', 'Wardweed'),
 (19, '1', 'Uncommon', 'Wardweed'),
 (20, '1', 'Rare', 'Wardweed'),
 (21, '5', 'Uncommon', 'Wardweed'),
 (22, '1', 'Common', 'Lifeleaf'),
 (22, '1', 'Common', 'Wardweed'),
 (23, '1', 'Uncommon', 'Lifeleaf'),
 (23, '1', 'Uncommon', 'Wardweed'),
 (24, '1', 'Rare', 'Lifeleaf'),
 (24, '1', 'Rare', 'Wardweed'),
 (25, '1', 'Common', 'Saltvine'),
 (26, '1

In [ ]:
if __name__ == "__main__":
    conn = sql.connect_pc()
    cursor = conn.cursor()

    drop_recipes_table()
    create_recipes_table()
    insert_recipes_table()

    conn.commit()
    conn.close()

parsing Alchemist
parsing Armourer
parsing Goldsmith
parsing Leathersmith
parsing Tailor
parsing Weaponsmith
parsing Woodsman


In [ ]:
conn = sql.connect_pc()
cursor = conn.cursor()

drop_ingredients_table()
create_ingredients_table()
insert_ingredients_table()

conn.commit()
conn.close()